# Modélisation — Prédiction du prix au m² des appartements parisiens

Ce notebook entraîne et compare trois modèles de régression sur le dataset DVF enrichi :
1. **Régression Linéaire** — baseline interprétable
2. **Random Forest** — ensemble robuste, non-linéaire
3. **Gradient Boosting** — gradient boosting, état de l'art sur données tabulaires

**Métriques d'évaluation :** MAE (€/m²), RMSE (€/m²), R²

## 1. Imports et configuration

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

sys.path.append("../src")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

from data import load_dataset_split
from metrics import compute_metrics

# Style des graphiques
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ Imports OK")

## 2. Chargement des données

In [ ]:
X_train, X_test, y_train, y_test = load_dataset_split()

print(f"X_train : {X_train.shape[0]:,} lignes × {X_train.shape[1]} features")
print(f"X_test  : {X_test.shape[0]:,} lignes × {X_test.shape[1]} features")
print(f"\nPrix médian train : {y_train.median():,.0f} €/m²")
print(f"Prix médian test  : {y_test.median():,.0f} €/m²")
X_train.head(3)

## 3. Modèle 1 — Régression Linéaire (baseline)

La régression linéaire suppose que le prix est une combinaison **linéaire** des features.
On utilise un `RobustScaler` dans un pipeline sklearn pour normaliser les features avant l'entraînement (le scaler est fitté uniquement sur le train set).

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", LinearRegression()),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

metrics_lr = compute_metrics(y_test, y_pred_lr)
print("=== Régression Linéaire ===")
print(f"MAE  : {metrics_lr['mae']:,.0f} €/m²")
print(f"RMSE : {metrics_lr['rmse']:,.0f} €/m²")
print(f"R²   : {metrics_lr['r2']:.4f}")

joblib.dump(lr_pipeline, "../models/linear_regression.pkl")
print("\n✅ Modèle sauvegardé dans models/linear_regression.pkl")

## 4. Modèle 2 — Random Forest

Le Random Forest entraîne 200 arbres de décision sur des sous-échantillons aléatoires et fait la moyenne de leurs prédictions. Pas besoin de scaler les features (les arbres sont insensibles à l'échelle).

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

metrics_rf = compute_metrics(y_test, y_pred_rf)
print("=== Random Forest ===")
print(f"MAE  : {metrics_rf['mae']:,.0f} €/m²")
print(f"RMSE : {metrics_rf['rmse']:,.0f} €/m²")
print(f"R²   : {metrics_rf['r2']:.4f}")

joblib.dump(rf_model, "../models/random_forest.pkl")
print("\n✅ Modèle sauvegardé dans models/random_forest.pkl")

## 5. Modèle 3 — Gradient Boosting (HistGradientBoostingRegressor)

Le Gradient Boosting construit les arbres **séquentiellement** : chaque arbre corrige les erreurs du précédent. `HistGradientBoostingRegressor` est l'implémentation moderne et rapide de scikit-learn, comparable à XGBoost en performance.

In [ ]:
gb_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

metrics_gb = compute_metrics(y_test, y_pred_gb)
print("=== Gradient Boosting ===")
print(f"MAE  : {metrics_gb['mae']:,.0f} €/m²")
print(f"RMSE : {metrics_gb['rmse']:,.0f} €/m²")
print(f"R²   : {metrics_gb['r2']:.4f}")

joblib.dump(gb_model, "../models/gradient_boosting.pkl")
print("\n✅ Modèle sauvegardé dans models/gradient_boosting.pkl")

## 6. Comparaison des modèles

In [ ]:
results = pd.DataFrame([
    {"Modèle": "Régression Linéaire", **metrics_lr},
    {"Modèle": "Random Forest",       **metrics_rf},
    {"Modèle": "Gradient Boosting",   **metrics_gb},
]).set_index("Modèle")

results.columns = ["MAE (€/m²)", "RMSE (€/m²)", "R²"]
results = results.round({"MAE (€/m²)": 0, "RMSE (€/m²)": 0, "R²": 4})

print("=== Comparaison des modèles (test set) ===")
display(results)

# Sauvegarde pour l'app Streamlit
import os
os.makedirs("../results", exist_ok=True)
results.reset_index().to_csv("../results/model_metrics.csv", index=False)
print("\n✅ Résultats sauvegardés dans results/model_metrics.csv")

## 7. Visualisations

### 7.1 Comparaison des MAE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

modeles = ["Régression Linéaire", "Random Forest", "Gradient Boosting"]
couleurs = ["#4C72B0", "#DD8452", "#55A868"]
maes   = [metrics_lr["mae"],  metrics_rf["mae"],  metrics_gb["mae"]]
rmses  = [metrics_lr["rmse"], metrics_rf["rmse"], metrics_gb["rmse"]]
r2s    = [metrics_lr["r2"],   metrics_rf["r2"],   metrics_gb["r2"]]

# MAE
axes[0].bar(modeles, maes, color=couleurs)
axes[0].set_title("MAE (€/m²) — moins c'est mieux", fontsize=12)
axes[0].set_ylabel("€/m²")
for i, v in enumerate(maes):
    axes[0].text(i, v + 10, f"{v:,.0f}", ha="center", fontweight="bold")

# RMSE
axes[1].bar(modeles, rmses, color=couleurs)
axes[1].set_title("RMSE (€/m²) — moins c'est mieux", fontsize=12)
axes[1].set_ylabel("€/m²")
for i, v in enumerate(rmses):
    axes[1].text(i, v + 10, f"{v:,.0f}", ha="center", fontweight="bold")

# R²
axes[2].bar(modeles, r2s, color=couleurs)
axes[2].set_title("R² — plus c'est mieux", fontsize=12)
axes[2].set_ylabel("R²")
axes[2].set_ylim(0, 1)
for i, v in enumerate(r2s):
    axes[2].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")

plt.suptitle("Comparaison des modèles — Test set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/comparaison_modeles.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé dans plots/comparaison_modeles.png")

### 7.2 Prédictions vs Réels (Gradient Boosting — meilleur modèle)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter : réels vs prédits
sample = np.random.choice(len(y_test), size=3000, replace=False)
axes[0].scatter(y_test.iloc[sample], y_pred_gb[sample], alpha=0.3, s=10, color="#55A868")
axes[0].plot([3000, 25000], [3000, 25000], "r--", linewidth=1.5, label="Prédiction parfaite")
axes[0].set_xlabel("Prix réel (€/m²)")
axes[0].set_ylabel("Prix prédit (€/m²)")
axes[0].set_title("Gradient Boosting — Réels vs Prédits")
axes[0].legend()

# Distribution des résidus
residus = y_test.values - y_pred_gb
axes[1].hist(residus, bins=80, color="#55A868", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Résidu (€/m²)")
axes[1].set_ylabel("Fréquence")
axes[1].set_title("Distribution des résidus — Gradient Boosting")

plt.suptitle("Analyse des prédictions Gradient Boosting", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/predictions_vs_reels.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Feature Importances (Gradient Boosting — permutation importance)

In [ ]:
# Permutation importance : on mélange une feature à la fois et on mesure la dégradation du R²
perm = permutation_importance(
    gb_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

importances = pd.Series(perm.importances_mean, index=X_train.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors = ["#55A868" if v >= importances.quantile(0.75) else "#AEC6CF" for v in importances]
importances.plot(kind="barh", color=colors)
plt.xlabel("Importance (baisse de R² si la feature est mélangée)")
plt.title("Feature Importances — Gradient Boosting", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 features les plus importantes :")
for feat, val in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat:<40} {val:.4f}")

## 8. Conclusion

Le meilleur modèle est **Gradient Boosting** avec les performances suivantes sur le test set :

- **MAE** : à remplir après exécution (€/m²)
- **RMSE** : à remplir après exécution (€/m²)
- **R²** : à remplir après exécution

### Interprétation business
- L'arrondissement, la latitude/longitude et la surface sont les variables les plus importantes
- Les features géographiques (distance aux stations, espaces verts) contribuent à affiner les prédictions
- Le Gradient Boosting améliore significativement la baseline linéaire grâce à sa capacité à modéliser les effets non-linéaires du marché immobilier parisien

### Limites
- Le modèle ne dispose pas d'informations sur l'étage, l'état du bien, la présence d'un balcon ou d'une cave
- Les coordonnées DVF sont approximatives (centroïde cadastral)
- Les données géographiques externes représentent l'état actuel, pas celui au moment des transactions